# Spectra-Sense held-out evaluation

Run from the repository root or the ml directory with the project virtual environment. The test tensors, labels and label encoder default to the repository inputs directory. Set SPECTRASENSE_INPUTS to use a different directory. The evaluator uses the model path from ml/config.py and compares fresh results with the committed metrics. Generated files stay in a temporary directory.

In [1]:
import json
import os
from pathlib import Path
import subprocess
import sys
from tempfile import TemporaryDirectory

working_dir = Path.cwd()
project = working_dir if (working_dir / "ml" / "config.py").is_file() else working_dir.parent
if not (project / "ml" / "config.py").is_file():
    raise FileNotFoundError("Run this notebook from the repository root or ml directory.")

input_dir = Path(os.environ.get("SPECTRASENSE_INPUTS", project / "inputs")).resolve()
expected = json.loads((project / "artifacts" / "metrics.json").read_text(encoding="utf-8"))

with TemporaryDirectory(prefix="spectrasense-evaluation-") as output_dir:
    result = subprocess.run(
        [
            sys.executable, "-m", "ml.evaluate",
            "--x-test", str(input_dir / "X_test.npy"),
            "--y-test", str(input_dir / "y_test.npy"),
            "--le-path", str(input_dir / "label_encoder.pkl"),
            "--output-dir", output_dir,
        ],
        cwd=project,
        capture_output=True,
        text=True,
        encoding="utf-8",
    )
    if result.returncode:
        raise RuntimeError(result.stderr or result.stdout)
    observed = json.loads((Path(output_dir) / "metrics.json").read_text(encoding="utf-8"))

In [2]:
keys = ("coal_to_coal", "rock_to_rock", "coal_to_rock", "rock_to_coal")

def comparison(metrics):
    return {
        "accuracy": metrics["multi_class"]["accuracy"],
        "error_decomposition": {key: metrics["error_decomposition"][key] for key in keys},
    }

observed_summary = comparison(observed)
expected_summary = comparison(expected)
print("Executed:", json.dumps(observed_summary))
print("Artifact:", json.dumps(expected_summary))
if observed_summary != expected_summary:
    raise AssertionError("Executed accuracy or error decomposition differs from artifacts/metrics.json.")
print("PASS: exact match for accuracy and all four error-decomposition counts.")

Executed: {"accuracy": 0.9572158365261814, "error_decomposition": {"coal_to_coal": 741, "rock_to_rock": 820, "coal_to_rock": 4, "rock_to_coal": 1}}
Artifact: {"accuracy": 0.9572158365261814, "error_decomposition": {"coal_to_coal": 741, "rock_to_rock": 820, "coal_to_rock": 4, "rock_to_coal": 1}}
PASS: exact match for accuracy and all four error-decomposition counts.
